# Decision Tree

A tree that asks you progressively more specific YES/NO questions until it figures out what you are. It's literally how a detective (or a five-year-old) solves mysteries!


## 1) Definition: What is a Decision Tree?

A **Decision Tree** is a supervised learning algorithm that makes predictions by recursively partitioning the data based on feature values. It creates a tree-like model of decisions to predict outcomes.

**Key Characteristics:**
- Hierarchical structure (flows top to bottom)
- Each node asks a question (condition)
- Each branch represents a possible answer
- Leaf nodes contain final predictions
- Human interpretable (can explain every decision)

**Why "Tree"?**
- Root node at the top (main question)
- Internal nodes branch based on conditions
- Leaf nodes are the final outcomes
- Actually it's drawn upside down from a real tree!

**Real-world analogy:**
Imagine a flowchart for loan approval:
- "Age > 30?" → If YES, ask next question. If NO, go to another branch.
- "Income > $50k?" → If YES, "APPROVED!" If NO, "REJECTED!"
- It's literally 20 Questions but for machine learning!


![Decision Tree Concept](images/dt_tree_concept.png)

**Image explanation:**
Follow the tree from top to bottom:
- Start at the red ROOT node: "Age > 30?"
- If YES (green): go to left branch, ask "Income > $50k?"
- If NO (red): go to right branch, ask "Has Credit Card?"
- Eventually reach a LEAF node with a decision: APPROVED or REJECTED

Each path through the tree is a different decision rule!


## 2) The Process: Growing a Tree

### Step 1: Start with the Root Node
The algorithm selects the BEST feature to split on (more on this later).

### Step 2: Recursive Splitting
For each subset created, repeat the process:
- Calculate which feature creates the best split
- Split the data on that feature
- Create child nodes

### Step 3: Stop When?
- All examples in a node belong to one class (PURE node)
- No more features to split on
- Maximum tree depth is reached (to prevent overfitting)
- Minimum samples per node is reached

### Step 4: Make Predictions
For a new example:
- Start at root
- Follow the splits based on feature values
- Reach a leaf node
- Return the majority class of that leaf


## 3) Tree Structure: Root, Branch, and Leaf Nodes

### Root Node
- The TOP of the tree
- Contains the FIRST split (the most important question)
- Asks the question that divides the data most effectively
- Only ONE root node per tree

### Branch Nodes (Internal Nodes)
- Between root and leaves
- Each asks a YES/NO question
- Splits data into subsets based on feature conditions
- Can have multiple levels
- Example: "Age > 30?" or "Income > $50k?"

### Leaf Nodes (Terminal Nodes)
- BOTTOM of the tree
- No more splits happen here
- Contains the FINAL PREDICTION
- For classification: the majority class in that leaf
- For regression: the average value in that leaf
- Examples: "APPROVED", "REJECTED", "HIGH RISK", etc.

### Depth of the Tree
- Shallow trees: simple, interpretable, but may underfit
- Deep trees: complex, fit training data perfectly, but may overfit
- Finding balance = key to good performance


## 4) How to Choose the Root Node and Best Splits

The algorithm must decide: "Which feature should I ask about FIRST?"

Answer: The feature that creates the most PURE subsets (least disorder).

### Entropy: Measuring Chaos

Before splitting, we calculate how MIXED the data is using **Entropy**:

`H(S) = -Σ p_i * log2(p_i)`

Where:
- `S` = the current dataset
- `p_i` = proportion of class i
- Σ = sum over all classes

**Interpretation:**
- H = 0: PURE (all one class)
- H = 1: MAXIMUM CHAOS (50-50 split for binary)
- H > 1: Possible for multi-class (more classes = more chaos possible)

### Example Entropy Calculation

Dataset: 8 Apples, 2 Oranges (binary classification)

```
p_apple = 8/10 = 0.8
p_orange = 2/10 = 0.2

H = -(0.8 * log2(0.8) + 0.2 * log2(0.2))
  = -(0.8 * (-0.322) + 0.2 * (-2.322))
  = -(-0.258 - 0.464)
  = 0.722
```

This dataset has SOME order (mostly apples) but not pure (has oranges).

### Information Gain: How Much Better is the Split?

**Information Gain (IG)** measures how much PURITY we gain by splitting:

`IG(S, A) = H(S) - Σ |S_v| / |S| * H(S_v)`

Where:
- `H(S)` = entropy of parent node
- `|S_v|` = size of child v
- `|S|` = size of parent
- `H(S_v)` = entropy of child v

**Translation:** "Parent entropy minus the average entropy of children"

**Key insight:**
- HIGH information gain = GOOD SPLIT (children are much purer)
- LOW information gain = BAD SPLIT (children are still messy)

**Algorithm chooses the feature with the HIGHEST information gain!**


![Entropy Example](images/dt_entropy_example.png)

**Image explanation:**
- Pure node (all red): Entropy = 0 (no chaos)
- Mixed nodes: Entropy increases (more colors = more chaos)
- Maximum chaos: Entropy = 1.0 (perfect 50-50 split)

Decision trees want to MINIMIZE entropy by choosing good splits!


![Information Gain](images/dt_information_gain.png)

**Image explanation:**
- LEFT (Bad split): Children are still messy. Info gain ≈ 0.07 (waste)
- RIGHT (Good split): Children are pure! Info gain ≈ 0.66 (excellent)

Decision trees choose splits with HIGH information gain!


## 5) The Algorithm (Pseudocode)

```
function BuildTree(data, features):
    # Base cases (stopping conditions)
    if all examples in data belong to same class:
        return Leaf(majority_class)
    
    if no features left to split:
        return Leaf(majority_class)
    
    if data is empty:
        return Leaf(default_class)
    
    # Recursive case
    best_feature = None
    best_gain = 0
    
    for each feature in features:
        for each possible split value:
            gain = calculate_information_gain(data, feature, split_value)
            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_split = split_value
    
    # Create node and recursively build left/right subtrees
    node = BranchNode(best_feature, best_split)
    left_data = data where feature <= best_split
    right_data = data where feature > best_split
    
    node.left = BuildTree(left_data, features)
    node.right = BuildTree(right_data, features)
    
    return node
```


## 6) Worked Example: Predicting Loan Approval

**Problem:** Predict if a customer should be approved for a loan based on age and income.

### Training Dataset:

```
| ID | Age | Income   | Approved |
|----+-----+----------+----------|
| 1  | 25  | $30k     | NO       |
| 2  | 35  | $60k     | YES      |
| 3  | 28  | $35k     | NO       |
| 4  | 45  | $70k     | YES      |
| 5  | 50  | $80k     | YES      |
| 6  | 23  | $25k     | NO       |
| 7  | 52  | $90k     | YES      |
| 8  | 30  | $55k     | NO       |
| 9  | 40  | $75k     | YES      |
| 10 | 26  | $32k     | NO       |
```

### Step 1: Calculate Initial Entropy

```
Total: 10 customers
Approved (YES): 5
Rejected (NO): 5

Parent Entropy:
H = -(0.5 * log2(0.5) + 0.5 * log2(0.5))
  = -(0.5 * (-1) + 0.5 * (-1))
  = 1.0 (Maximum chaos! Perfectly mixed)
```

### Step 2: Try Splitting on "Age > 35"

```
LEFT (Age <= 35): IDs 1, 3, 6, 8, 10
  - YES: 0
  - NO: 5
  - Entropy = 0 (PURE!)

RIGHT (Age > 35): IDs 2, 4, 5, 7, 9
  - YES: 5
  - NO: 0
  - Entropy = 0 (PURE!)
```

### Step 3: Calculate Information Gain

```
Average Child Entropy:
= (5/10) * 0 + (5/10) * 0
= 0

Information Gain = Parent - Weighted Children
= 1.0 - 0
= 1.0 (PERFECT! Maximum possible gain)
```

### Step 4: Root Node Decision

Since "Age > 35" gives us IG = 1.0 (maximum), it becomes the ROOT NODE!

```
           [Age > 35?]
          /           \
       NO/             \YES
      /                 \
   [REJECTED]        [APPROVED]
   (All 5 NO)         (All 5 YES)
```

### Step 5: Prediction on New Data

**New Customer:** Age = 40, Income = $65k

```
Start at root: "Age > 35?"
40 > 35? YES → Go RIGHT
Reach leaf: APPROVED

PREDICTION: Approve the loan!
```

**Another Customer:** Age = 28, Income = $50k

```
Start at root: "Age > 35?"
28 > 35? NO → Go LEFT
Reach leaf: REJECTED

PREDICTION: Reject the application.
```

### Key Insights:

1. A single split on Age perfectly separates our classes!
2. Information gain was MAXIMUM (1.0), so this is the best root
3. Both child nodes are pure (entropy = 0)
4. The tree is very shallow (just 1 level)
5. We can explain every decision: "We approve anyone over 35!"


## Summary: Decision Trees in a Nutshell

**Pros:**
- ✓ Highly interpretable (can explain decisions)
- ✓ No feature scaling needed
- ✓ Handles both classification and regression
- ✓ Works with non-linear relationships
- ✓ Fast predictions (just follow branches)

**Cons:**
- ✗ Prone to overfitting (deep trees memorize noise)
- ✗ Unstable (small data changes = big tree changes)
- ✗ Biased toward high-cardinality features
- ✗ Doesn't handle missing values well

**Best for:**
- Classification and regression
- When interpretability is critical
- Business decision rules ("If X then Y")
- Medical diagnosis (doctors understand decisions)

**The Bottom Line:**
Decision Trees are the "training wheels" of machine learning. They're simple, interpretable, and effective for many problems. But they need careful pruning to avoid overfitting!
